In [ ]:
from google.colab import drive
drive.mount('/content/drive')

unzip the file

In [ ]:
import os

dataset = "/content/drive/MyDrive/IndividualProject/cxr"

if not os.path.exists(dataset):
    !unzip "/content/drive/MyDrive/IndividualProject/cxr.zip" -d "/content/drive/MyDrive/IndividualProject/"
else:
    print("Dataset already extracted.")

In [ ]:
!ls /content/drive/MyDrive/IndividualProject/cxr

Understanding the data through annotations.csv.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/IndividualProject/cxr/annotations.csv")
print(df.shape)
print(df.columns)
df.head()

In [ ]:
!ls /content/drive/MyDrive/IndividualProject/cxr/task_labels

In [ ]:
import numpy as np

labels = np.load("/content/drive/MyDrive/IndividualProject/cxr/task_labels/disease labels.npy")
sex = np.load("/content/drive/MyDrive/IndividualProject/cxr/task_labels/patient sex.npy")

print(labels.shape, sex.shape)


In [ ]:
import yaml

with open("/content/drive/MyDrive/IndividualProject/cxr/info.yaml", "r") as f:
    info = yaml.safe_load(f)

info.keys()


In [ ]:
info

In [ ]:
# try to load images and verify it is 224x224 pixels
import h5py
import random

with h5py.File("/content/drive/MyDrive/IndividualProject/cxr/images.hdf5", "r") as f:
  print("Keys in HDF5", list(f.keys()))

  images = f['images']
  random_index = random.randint(0, len(images) - 1)
  random_image = images[random_index]

  print("Random index:", random_index)
  print("Image shape:", random_image.shape)

In [ ]:
print("Data type:", random_image.dtype)
print("Min/Max value:", random_image.min(), "/", random_image.max())

In [ ]:
# visualise the image
import matplotlib.pyplot as plt

plt.imshow(random_image, cmap='gray')
plt.show()

To verify the train-test-validation split

In [ ]:
!ls /content/drive/MyDrive/IndividualProject/cxr/splits

In [ ]:
with open("/content/drive/MyDrive/IndividualProject/cxr/splits/train.txt") as f:
    for _ in range(5):
        print(f.readline().strip())

In [ ]:
def load_split(path):
    with open(path, "r") as f:
        return set(line.strip() for line in f)

split_dir = "/content/drive/MyDrive/IndividualProject/cxr/splits"

train_ids = load_split(f"{split_dir}/train.txt")
val_ids   = load_split(f"{split_dir}/val.txt")
test_ids  = load_split(f"{split_dir}/test.txt")

print("Train size:", len(train_ids))
print("Val size:", len(val_ids))
print("Test size:", len(test_ids))


In [ ]:
# total number of cases
total = len(train_ids) + len(val_ids) + len(test_ids)
print("Total cases:", total)

In [ ]:
# verify the percentage of the split
train_pct = 100 * len(train_ids) / total
val_pct   = 100 * len(val_ids) / total
test_pct  = 100 * len(test_ids) / total

print(f"Train: {len(train_ids)} ({train_pct:.2f}%)")
print(f"Val:   {len(val_ids)} ({val_pct:.2f}%)")
print(f"Test:  {len(test_ids)} ({test_pct:.2f}%)")

In [ ]:
print("Train ∩ Val:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Val ∩ Test:", len(val_ids & test_ids))


To identify the total number of cases for each disease labels.

In [ ]:
import ast

disease_names = info['tasks'][0]['labels']
df["disease_vector"] = df["tasks/disease labels"].apply(ast.literal_eval)

In [ ]:
disease_counts = {}

for idx, name in disease_names.items():
    disease_counts[name] = df["disease_vector"].apply(lambda x: x[idx] == 1).sum()

In [ ]:
pd.DataFrame.from_dict(disease_counts, orient="index", columns=["positive_cases"]) \
  .sort_values("positive_cases", ascending=False)

Identify the distribution of male and female in the dataset.

In [ ]:
# identify the distribution of male and female
df['patient sex'].value_counts(normalize=True)

Identify the distribution of patients' age in the dataset.

In [ ]:
# the age are grouped into three (people less than 40 years old, patients between 40 to 64 years old and patients of 65 years old and above)
bins = [-float('inf'), 40, 65, float('inf')]
age_labels = ['<40', '40-65', '>65']

df['patient_age_group'] = pd.cut(df['patient_age'], bins=bins, labels=age_labels, right=False)

In [ ]:
df['patient_age_group'].value_counts()
df['patient_age_group'].value_counts(normalize=True)

Identify the count for each split - to identify the percentage of the split

In [ ]:
df['split'].value_counts(normalize=True)

In [ ]:
# check the distribution of patient's sex and age among each split
df.groupby('split')['patient sex'].value_counts(normalize=True)

In [ ]:
# 'normalize=index' means "calculate percentages across the rows"
result = pd.crosstab(df['split'], df['patient sex'], normalize='index')

In [ ]:
df.groupby('split')['patient_age_group'].value_counts(normalize=True)